In [1]:
# Verify Python environment and core packages
import sys
import pandas as pd
import numpy as np
import requests

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("Environment ready for TCGA analysis.")

Python: 3.11.17 (main, Oct  1 2026, 00:48:48) [Clang 16.0.0 (clang-1600.0.26.6)]
Pandas: 2.3.3
Environment ready for TCGA analysis.


In [2]:
# Retrieve TCGA-BRCA project metadata from the NCI GDC API

import requests

project_id = "TCGA-BRCA"

url = f"https://api.gdc.cancer.gov/projects/{project_id}"

response = requests.get(url)

print("Status code:", response.status_code)

Status code: 200


In [3]:
# Inspect key TCGA-BRCA project metadata returned by the GDC API

project_data = response.json()["data"]

print("Project ID:", project_data["project_id"])
print("Project Name:", project_data["name"])
print("Primary Site:", project_data["primary_site"])
print("Disease Type:", project_data["disease_type"])
print("Number of Cases:", project_data["summary"]["case_count"])

Project ID: TCGA-BRCA
Project Name: Breast Invasive Carcinoma
Primary Site: ['Breast']
Disease Type: ['Cystic, Mucinous and Serous Neoplasms', 'Adenomas and Adenocarcinomas', 'Fibroepithelial Neoplasms', 'Basal Cell Neoplasms', 'Ductal and Lobular Neoplasms', 'Epithelial Neoplasms, NOS', 'Squamous Cell Neoplasms', 'Complex Epithelial Neoplasms', 'Adnexal and Skin Appendage Neoplasms']


KeyError: 'summary'

In [4]:
# Inspect the keys returned for TCGA-BRCA project metadata

print(project_data.keys())

dict_keys(['primary_site', 'dbgap_accession_number', 'project_id', 'disease_type', 'name', 'releasable', 'state', 'released'])


In [5]:
# Count the total number of TCGA-BRCA cases using the GDC cases endpoint

import json

cases_url = "https://api.gdc.cancer.gov/cases"

filters = {
    "op": "=",
    "content": {
        "field": "project.project_id",
        "value": "TCGA-BRCA"
    }
}

params = {
    "filters": json.dumps(filters),
    "format": "JSON",
    "size": 1
}

cases_response = requests.get(cases_url, params=params)

print("Status code:", cases_response.status_code)

Status code: 200


In [6]:
# Read the total number of TCGA-BRCA cases from the API response

cases_data = cases_response.json()

total_cases = cases_data["data"]["pagination"]["total"]

print("Total TCGA-BRCA cases:", total_cases)

Total TCGA-BRCA cases: 1098


In [7]:
# Retrieve the first 10 TCGA-BRCA case identifiers

case_params = {
    "filters": json.dumps(filters),
    "fields": "case_id,submitter_id",
    "format": "JSON",
    "size": 10
}

case_response = requests.get(cases_url, params=case_params)

print("Status code:", case_response.status_code)

Status code: 200


In [8]:
# Inspect the first TCGA-BRCA case returned by the GDC API

case_data = case_response.json()["data"]["hits"]

print("Number of cases retrieved:", len(case_data))
print()
print("First case:")
print(case_data[0])

Number of cases retrieved: 10

First case:
{'id': '57a1604c-60b7-4b30-a75e-f70939532c5c', 'case_id': '57a1604c-60b7-4b30-a75e-f70939532c5c', 'submitter_id': 'TCGA-BH-A0B2'}


In [9]:
# Convert the retrieved TCGA-BRCA cases into a pandas DataFrame

cases_df = pd.DataFrame(case_data)

cases_df

,id,case_id,submitter_id
0,57a1604c-60b7-4b30-a75e-f70939532c5c,57a1604c-60b7-4b30-a75e-f70939532c5c,TCGA-BH-A0B2
1,2779fa01-ac93-4e80-a997-3385f72172c3,2779fa01-ac93-4e80-a997-3385f72172c3,TCGA-A8-A08S
2,80f3f48a-cc21-415c-b181-c77e7ba1c563,80f3f48a-cc21-415c-b181-c77e7ba1c563,TCGA-A8-A09C
3,e663954c-05a8-479a-a623-400c1eb5efc6,e663954c-05a8-479a-a623-400c1eb5efc6,TCGA-BH-A1F5
4,14b95463-2108-4921-afc2-e29eef52b18f,14b95463-2108-4921-afc2-e29eef52b18f,TCGA-A2-A0YT
5,fa4e082a-d213-412e-8d91-842a6f4bef12,fa4e082a-d213-412e-8d91-842a6f4bef12,TCGA-BH-A0B6
6,02f5ae33-a563-4ecb-9e33-dfa500a44931,02f5ae33-a563-4ecb-9e33-dfa500a44931,TCGA-A8-A093
7,3dbe99d1-e3b8-4ee2-b6a8-2e2e12c6fbe9,3dbe99d1-e3b8-4ee2-b6a8-2e2e12c6fbe9,TCGA-A8-A084
8,dfefb76a-ec6b-4cd2-9d45-2e1e4befc7ea,dfefb76a-ec6b-4cd2-9d45-2e1e4befc7ea,TCGA-A8-A09E
9,cb9f5e50-f49d-4899-8895-9367afcc1015,cb9f5e50-f49d-4899-8895-9367afcc1015,TCGA-A7-A0DC


In [10]:
# Retrieve all TCGA-BRCA case identifiers

all_case_params = {
    "filters": json.dumps(filters),
    "fields": "case_id,submitter_id",
    "format": "JSON",
    "size": total_cases
}

all_case_response = requests.get(cases_url, params=all_case_params)

print("Status code:", all_case_response.status_code)

Status code: 200


In [11]:
# Convert all TCGA-BRCA case identifiers into a pandas DataFrame

all_case_data = all_case_response.json()["data"]["hits"]

all_cases_df = pd.DataFrame(all_case_data)

print("Number of rows:", len(all_cases_df))

all_cases_df.head()

Number of rows: 1098


,id,case_id,submitter_id
0,57a1604c-60b7-4b30-a75e-f70939532c5c,57a1604c-60b7-4b30-a75e-f70939532c5c,TCGA-BH-A0B2
1,2779fa01-ac93-4e80-a997-3385f72172c3,2779fa01-ac93-4e80-a997-3385f72172c3,TCGA-A8-A08S
2,80f3f48a-cc21-415c-b181-c77e7ba1c563,80f3f48a-cc21-415c-b181-c77e7ba1c563,TCGA-A8-A09C
3,e663954c-05a8-479a-a623-400c1eb5efc6,e663954c-05a8-479a-a623-400c1eb5efc6,TCGA-BH-A1F5
4,14b95463-2108-4921-afc2-e29eef52b18f,14b95463-2108-4921-afc2-e29eef52b18f,TCGA-A2-A0YT


In [12]:
# Check whether TCGA-BRCA case identifiers are unique

print("Total rows:", len(all_cases_df))
print("Unique case_id values:", all_cases_df["case_id"].nunique())
print("Unique submitter_id values:", all_cases_df["submitter_id"].nunique())

Total rows: 1098
Unique case_id values: 1098
Unique submitter_id values: 1098


In [13]:
# Retrieve basic clinical information for TCGA-BRCA cases

clinical_fields = [
    "case_id",
    "submitter_id",
    "demographic.gender",
    "demographic.vital_status",
    "diagnoses.age_at_diagnosis",
    "diagnoses.tumor_stage"
]

clinical_params = {
    "filters": json.dumps(filters),
    "fields": ",".join(clinical_fields),
    "format": "JSON",
    "size": total_cases
}

clinical_response = requests.get(cases_url, params=clinical_params)

print("Status code:", clinical_response.status_code)

Status code: 200


In [14]:
# Inspect one TCGA-BRCA clinical record returned by the GDC API

clinical_data = clinical_response.json()["data"]["hits"]

print("Number of clinical records retrieved:", len(clinical_data))
print()
print(clinical_data[0])

Number of clinical records retrieved: 1098

{'id': '57a1604c-60b7-4b30-a75e-f70939532c5c', 'case_id': '57a1604c-60b7-4b30-a75e-f70939532c5c', 'submitter_id': 'TCGA-BH-A0B2'}


In [15]:
# Retrieve TCGA-BRCA cases with expanded demographic and diagnosis information

clinical_params = {
    "filters": json.dumps(filters),
    "fields": "case_id,submitter_id",
    "expand": "demographic,diagnoses",
    "format": "JSON",
    "size": total_cases
}

clinical_response = requests.get(cases_url, params=clinical_params)

print("Status code:", clinical_response.status_code)

Status code: 200


In [16]:
# Inspect one TCGA-BRCA case with expanded clinical information

clinical_data = clinical_response.json()["data"]["hits"]

print(clinical_data[0])

{'id': '57a1604c-60b7-4b30-a75e-f70939532c5c', 'case_id': '57a1604c-60b7-4b30-a75e-f70939532c5c', 'submitter_id': 'TCGA-BH-A0B2'}


In [17]:
# Retrieve TCGA-BRCA clinical data using a POST request with expanded fields

clinical_payload = {
    "filters": filters,
    "fields": "case_id,submitter_id,demographic.gender,demographic.vital_status,diagnoses.age_at_diagnosis,diagnoses.tumor_stage",
    "expand": "demographic,diagnoses",
    "format": "JSON",
    "size": total_cases
}

clinical_response = requests.post(
    cases_url,
    json=clinical_payload
)

print("Status code:", clinical_response.status_code)

Status code: 200


In [18]:
# Inspect one TCGA-BRCA case returned from the POST clinical request

clinical_data = clinical_response.json()["data"]["hits"]

print(clinical_data[0])

{'id': '57a1604c-60b7-4b30-a75e-f70939532c5c', 'case_id': '57a1604c-60b7-4b30-a75e-f70939532c5c', 'submitter_id': 'TCGA-BH-A0B2'}


In [19]:
# Check whether the GDC API returned any warnings about our clinical fields

clinical_json = clinical_response.json()

print(clinical_json.get("warnings"))

{'fields': 'unrecognized values: [demographic.gender, diagnoses.tumor_stage]'}


In [20]:
# Retrieve TCGA-BRCA clinical data using valid GDC field names

clinical_payload = {
    "filters": filters,
    "fields": (
        "case_id,submitter_id,"
        "demographic.sex_at_birth,"
        "demographic.vital_status,"
        "diagnoses.age_at_diagnosis,"
        "diagnoses.ajcc_pathologic_stage"
    ),
    "expand": "demographic,diagnoses",
    "format": "JSON",
    "size": total_cases
}

clinical_response = requests.post(
    cases_url,
    json=clinical_payload
)

print("Status code:", clinical_response.status_code)
print("Warnings:", clinical_response.json().get("warnings"))

Status code: 200
Warnings: {}


In [21]:
# Inspect one TCGA-BRCA case with valid clinical fields

clinical_data = clinical_response.json()["data"]["hits"]

print(clinical_data[0])

{'id': '57a1604c-60b7-4b30-a75e-f70939532c5c', 'case_id': '57a1604c-60b7-4b30-a75e-f70939532c5c', 'submitter_id': 'TCGA-BH-A0B2'}


In [22]:
# Retrieve TCGA-BRCA cases with full demographic and diagnosis objects

clinical_payload = {
    "filters": filters,
    "fields": "case_id,submitter_id",
    "expand": "demographic,diagnoses",
    "format": "JSON",
    "size": total_cases
}

clinical_response = requests.post(
    cases_url,
    json=clinical_payload
)

print("Status code:", clinical_response.status_code)
print("Warnings:", clinical_response.json().get("warnings"))

Status code: 200
Warnings: {}


In [23]:
# Inspect one TCGA-BRCA case after expanding demographic and diagnosis data

clinical_data = clinical_response.json()["data"]["hits"]

print(clinical_data[0])

{'id': '57a1604c-60b7-4b30-a75e-f70939532c5c', 'case_id': '57a1604c-60b7-4b30-a75e-f70939532c5c', 'submitter_id': 'TCGA-BH-A0B2'}


In [24]:
# Retrieve selected TCGA-BRCA clinical fields directly from the GDC cases endpoint

clinical_fields = [
    "case_id",
    "submitter_id",
    "demographic.sex_at_birth",
    "demographic.vital_status",
    "diagnoses.age_at_diagnosis",
    "diagnoses.ajcc_pathologic_stage"
]

clinical_params = {
    "filters": json.dumps(filters),
    "fields": ",".join(clinical_fields),
    "format": "JSON",
    "size": total_cases
}

clinical_response = requests.get(
    cases_url,
    params=clinical_params
)

print("Status code:", clinical_response.status_code)
print("Warnings:", clinical_response.json().get("warnings"))

Status code: 200
Warnings: {}


In [25]:
# Inspect one TCGA-BRCA case returned with direct clinical fields

clinical_data = clinical_response.json()["data"]["hits"]

print(clinical_data[0])

{'id': '57a1604c-60b7-4b30-a75e-f70939532c5c', 'case_id': '57a1604c-60b7-4b30-a75e-f70939532c5c', 'submitter_id': 'TCGA-BH-A0B2'}


In [26]:
# Retrieve one TCGA-BRCA patient directly using the GDC case UUID

first_case_id = all_cases_df.loc[0, "case_id"]

single_case_url = f"https://api.gdc.cancer.gov/cases/{first_case_id}"

single_case_response = requests.get(
    single_case_url,
    params={
        "expand": "demographic,diagnoses",
        "pretty": "true"
    }
)

print("Status code:", single_case_response.status_code)
print()
print(single_case_response.json()["data"])

Status code: 200

{'slide_ids': ['643fd386-3bcb-4ee1-a030-1e34eeefeb04', 'b3643e0d-dc70-4fd2-a0e0-534890f73da1'], 'submitter_slide_ids': ['TCGA-BH-A0B2-01A-01-TSA', 'TCGA-BH-A0B2-01A-01-BSA'], 'disease_type': 'Ductal and Lobular Neoplasms', 'analyte_ids': ['97f3d921-4fe1-40ad-9f84-e0537372ad3f', 'c7efa2f3-6724-4cc6-a375-4c6026c019e9', 'e4b5ef86-2bda-4498-a4f0-42d1a0a65a05', '41054d29-473d-4db6-9523-feb4000486a2', 'fe787c20-f983-4e10-a08b-72a48d07e303'], 'submitter_id': 'TCGA-BH-A0B2', 'submitter_analyte_ids': ['TCGA-BH-A0B2-01A-11R', 'TCGA-BH-A0B2-01A-11D', 'TCGA-BH-A0B2-10A-01D', 'TCGA-BH-A0B2-01A-11W', 'TCGA-BH-A0B2-10A-01W'], 'aliquot_ids': ['30143b3f-7a18-446f-b3b9-18a6748d7155', '760f4cd1-ad18-4acd-a1c9-a9c3f396658d', '0a8c0095-44df-4b5e-9479-4a9920692a06', 'a27c5907-1ad2-4414-a9c9-448f1b484946', 'de5fed7f-9770-4cb3-85bd-1b0bec7440da', '3a713c13-5bc4-4d9c-9e6f-85b2aaa8d85e', 'ae2856f4-dafc-45b6-96af-6c6615cf9475', 'd56e6268-4f07-457e-87f4-fb87d8d33c5a', '80c5eea0-7133-4894-a74a-21

In [27]:
# Retrieve clinical fields for one specific TCGA-BRCA case

single_case_filter = {
    "op": "=",
    "content": {
        "field": "case_id",
        "value": first_case_id
    }
}

single_clinical_params = {
    "filters": json.dumps(single_case_filter),
    "fields": (
        "case_id,submitter_id,"
        "demographic.sex_at_birth,"
        "demographic.vital_status,"
        "diagnoses.age_at_diagnosis,"
        "diagnoses.ajcc_pathologic_stage"
    ),
    "format": "JSON",
    "size": 1
}

single_clinical_response = requests.get(
    cases_url,
    params=single_clinical_params
)

print("Status code:", single_clinical_response.status_code)
print("Warnings:", single_clinical_response.json().get("warnings"))
print()
print(single_clinical_response.json()["data"]["hits"][0])

Status code: 200
Warnings: {}

{'id': '57a1604c-60b7-4b30-a75e-f70939532c5c', 'case_id': '57a1604c-60b7-4b30-a75e-f70939532c5c', 'submitter_id': 'TCGA-BH-A0B2'}


In [28]:
# Test clinical data retrieval using a different TCGA-BRCA patient

test_case_id = all_cases_df.loc[1, "case_id"]
test_submitter_id = all_cases_df.loc[1, "submitter_id"]

test_filter = {
    "op": "=",
    "content": {
        "field": "case_id",
        "value": test_case_id
    }
}

test_params = {
    "filters": json.dumps(test_filter),
    "fields": (
        "case_id,submitter_id,"
        "demographic.sex_at_birth,"
        "demographic.vital_status,"
        "diagnoses.age_at_diagnosis,"
        "diagnoses.ajcc_pathologic_stage"
    ),
    "format": "JSON",
    "size": 1
}

test_response = requests.get(
    cases_url,
    params=test_params
)

print("Testing patient:", test_submitter_id)
print("Status code:", test_response.status_code)
print()
print(test_response.json()["data"]["hits"][0])

Testing patient: TCGA-A8-A08S
Status code: 200

{'id': '2779fa01-ac93-4e80-a997-3385f72172c3', 'case_id': '2779fa01-ac93-4e80-a997-3385f72172c3', 'submitter_id': 'TCGA-A8-A08S', 'diagnoses': [{'age_at_diagnosis': None}, {'age_at_diagnosis': 25994, 'ajcc_pathologic_stage': 'Stage IIA'}], 'demographic': {'vital_status': 'Alive', 'sex_at_birth': 'female'}}


In [29]:
# Convert age at diagnosis from days to years for the test patient

age_days = 25994
age_years = age_days / 365.25

print("Age at diagnosis in years:", round(age_years, 1))

Age at diagnosis in years: 71.2


In [30]:
# Build a clean TCGA-BRCA clinical table from demographic and diagnosis records

clinical_rows = []

for case in clinical_response.json()["data"]["hits"]:
    case_id = case.get("case_id")
    submitter_id = case.get("submitter_id")

    demographic = case.get("demographic", {})
    sex_at_birth = demographic.get("sex_at_birth")
    vital_status = demographic.get("vital_status")

    diagnoses = case.get("diagnoses", [])

    age_at_diagnosis = None
    ajcc_pathologic_stage = None

    for diagnosis in diagnoses:
        if age_at_diagnosis is None and diagnosis.get("age_at_diagnosis") is not None:
            age_at_diagnosis = diagnosis.get("age_at_diagnosis")

        if ajcc_pathologic_stage is None and diagnosis.get("ajcc_pathologic_stage") is not None:
            ajcc_pathologic_stage = diagnosis.get("ajcc_pathologic_stage")

    clinical_rows.append({
        "case_id": case_id,
        "submitter_id": submitter_id,
        "sex_at_birth": sex_at_birth,
        "vital_status": vital_status,
        "age_at_diagnosis_days": age_at_diagnosis,
        "ajcc_pathologic_stage": ajcc_pathologic_stage
    })

clinical_df = pd.DataFrame(clinical_rows)

print("Clinical rows:", len(clinical_df))

clinical_df.head()

Clinical rows: 1098


,case_id,submitter_id,sex_at_birth,vital_status,age_at_diagnosis_days,ajcc_pathologic_stage
0,57a1604c-60b7-4b30-a75e-f70939532c5c,TCGA-BH-A0B2,None,None,NaN,None
1,2779fa01-ac93-4e80-a997-3385f72172c3,TCGA-A8-A08S,female,Alive,25994.0,Stage IIA
2,80f3f48a-cc21-415c-b181-c77e7ba1c563,TCGA-A8-A09C,female,Alive,25355.0,Stage X
3,e663954c-05a8-479a-a623-400c1eb5efc6,TCGA-BH-A1F5,female,Dead,22842.0,Stage IIA
4,14b95463-2108-4921-afc2-e29eef52b18f,TCGA-A2-A0YT,female,Dead,20482.0,Stage IIIB


In [31]:
# Convert age at diagnosis from days to years for all TCGA-BRCA cases

clinical_df["age_at_diagnosis_years"] = (
    clinical_df["age_at_diagnosis_days"] / 365.25
)

clinical_df["age_at_diagnosis_years"] = (
    clinical_df["age_at_diagnosis_years"].round(1)
)

clinical_df.head()

,case_id,submitter_id,sex_at_birth,vital_status,age_at_diagnosis_days,ajcc_pathologic_stage,age_at_diagnosis_years
0,57a1604c-60b7-4b30-a75e-f70939532c5c,TCGA-BH-A0B2,None,None,NaN,None,NaN
1,2779fa01-ac93-4e80-a997-3385f72172c3,TCGA-A8-A08S,female,Alive,25994.0,Stage IIA,71.2
2,80f3f48a-cc21-415c-b181-c77e7ba1c563,TCGA-A8-A09C,female,Alive,25355.0,Stage X,69.4
3,e663954c-05a8-479a-a623-400c1eb5efc6,TCGA-BH-A1F5,female,Dead,22842.0,Stage IIA,62.5
4,14b95463-2108-4921-afc2-e29eef52b18f,TCGA-A2-A0YT,female,Dead,20482.0,Stage IIIB,56.1


In [32]:
# Check missing values in the TCGA-BRCA clinical table

clinical_df[
    [
        "sex_at_birth",
        "vital_status",
        "age_at_diagnosis_years",
        "ajcc_pathologic_stage"
    ]
].isna().sum()

sex_at_birth               1
vital_status               1
age_at_diagnosis_years    16
ajcc_pathologic_stage     12
dtype: int64

In [33]:
# Inspect the categories present in key TCGA-BRCA clinical variables

print("Sex at birth:")
print(clinical_df["sex_at_birth"].value_counts(dropna=False))
print()

print("Vital status:")
print(clinical_df["vital_status"].value_counts(dropna=False))
print()

print("AJCC pathologic stage:")
print(clinical_df["ajcc_pathologic_stage"].value_counts(dropna=False))

Sex at birth:
sex_at_birth
female    1085
male        12
None         1
Name: count, dtype: int64

Vital status:
vital_status
Alive    945
Dead     152
None       1
Name: count, dtype: int64

AJCC pathologic stage:
ajcc_pathologic_stage
Stage IIA     357
Stage IIB     256
Stage IIIA    154
Stage I        89
Stage IA       87
Stage IIIC     64
Stage IIIB     27
Stage IV       20
Stage X        12
None           12
Stage II        7
Stage IB        7
Stage 0         4
Stage III       2
Name: count, dtype: int64


In [34]:
# Summarize age at diagnosis in the TCGA-BRCA cohort

clinical_df["age_at_diagnosis_years"].describe()

count    1082.000000
mean       59.301109
std        13.135294
min        26.900000
25%        49.525000
50%        59.150000
75%        68.300000
max        90.000000
Name: age_at_diagnosis_years, dtype: float64

In [35]:
# Save the cleaned TCGA-BRCA clinical table to the processed data folder

clinical_df.to_csv(
    "../data/processed/tcga_brca_clinical.csv",
    index=False
)

print("Saved clinical table successfully.")

Saved clinical table successfully.


In [36]:
# Verify the saved TCGA-BRCA clinical table can be loaded correctly

clinical_check = pd.read_csv("../data/processed/tcga_brca_clinical.csv")

print("Rows:", len(clinical_check))
print("Columns:", list(clinical_check.columns))

clinical_check.head()

Rows: 1098
Columns: ['case_id', 'submitter_id', 'sex_at_birth', 'vital_status', 'age_at_diagnosis_days', 'ajcc_pathologic_stage', 'age_at_diagnosis_years']


,case_id,submitter_id,sex_at_birth,vital_status,age_at_diagnosis_days,ajcc_pathologic_stage,age_at_diagnosis_years
0,57a1604c-60b7-4b30-a75e-f70939532c5c,TCGA-BH-A0B2,NaN,NaN,NaN,NaN,NaN
1,2779fa01-ac93-4e80-a997-3385f72172c3,TCGA-A8-A08S,female,Alive,25994.0,Stage IIA,71.2
2,80f3f48a-cc21-415c-b181-c77e7ba1c563,TCGA-A8-A09C,female,Alive,25355.0,Stage X,69.4
3,e663954c-05a8-479a-a623-400c1eb5efc6,TCGA-BH-A1F5,female,Dead,22842.0,Stage IIA,62.5
4,14b95463-2108-4921-afc2-e29eef52b18f,TCGA-A2-A0YT,female,Dead,20482.0,Stage IIIB,56.1
